
# Strander Predictive Maintenance — Incremental Jupyter Project

This notebook is designed to grow with your data over time.

You can place daily or monthly AVEVA Historian exports into a folder, then run the notebook to:

1. Import and combine long-format Excel or CSV files
2. Convert them into a clean wide-format time series
3. Build a normal-operation baseline
4. Detect anomalies
5. Add known failure dates later
6. Learn pre-failure patterns
7. Compare today's data to earlier failures

The notebook is divided into three phases so you do not need all historical data immediately.

---

## Expected AVEVA export format

The exports may look like:

| DateTime | TagName | Value | Other metadata... |
|---|---|---:|---|
| 7/1/2026 14:58 | A31A31DYNLeftCh0Overall | 0.0091 | ... |
| 7/1/2026 14:58 | A31A31DYNLeftCh1Overall | 0.0033 | ... |
| 7/1/2026 14:58 | A31Cps_ActSpd | 12.0 | ... |

The notebook automatically pivots this into one row per timestamp.

---

## Your sensor mapping

| Historian tag | Meaning |
|---|---|
| A31A31DYNLeftCh0Overall | Lower motor |
| A31A31DYNLeftCh1Overall | Upper gearbox |
| A31A31DYNLeftCh2Overall | Lower gearbox |
| A31A31DYNLeftCh3Overall | Motor-side bearings |
| A31A31DYNRightCh0Overall | Motor-side base |
| A31A31DYNRightCh1Overall | Non-motor-side bearings |
| A31A31DYNRightCh2Overall | Non-motor-side base |
| A31A31DYNRightCh3Overall | Upper motor |
| A31Cps_ActSpd | Actual speed |


## 0. Install packages if needed

In [ ]:

# Uncomment this line only if the packages are missing:
# %pip install pandas numpy matplotlib scikit-learn openpyxl joblib


## 1. Imports

In [ ]:

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor, IsolationForest, RandomForestClassifier
from sklearn.mixture import GaussianMixture
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)



## 2. Project folders and settings

Create this structure next to the notebook:

```text
strander_project/
├── data/
│   ├── historical/
│   ├── current/
│   └── failures.csv
├── processed/
├── models/
└── outputs/
```

Put daily or monthly AVEVA exports into `data/historical/`.

Put the newest file you want to evaluate into `data/current/`.


In [ ]:

PROJECT_ROOT = Path("strander_project")

HISTORICAL_DATA_DIR = PROJECT_ROOT / "data" / "historical"
CURRENT_DATA_DIR = PROJECT_ROOT / "data" / "current"
FAILURE_FILE = PROJECT_ROOT / "data" / "failures.csv"

PROCESSED_DIR = PROJECT_ROOT / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

for folder in [
    HISTORICAL_DATA_DIR,
    CURRENT_DATA_DIR,
    PROCESSED_DIR,
    MODELS_DIR,
    OUTPUTS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project created at:", PROJECT_ROOT.resolve())


## 3. Historian tag mapping

In [ ]:

TAG_MAP = {
    "A31A31DYNLeftCh0Overall": "LowerMotor",
    "A31A31DYNLeftCh1Overall": "UpperGearbox",
    "A31A31DYNLeftCh2Overall": "LowerGearbox",
    "A31A31DYNLeftCh3Overall": "MotorSideBearing",
    "A31A31DYNRightCh0Overall": "MotorSideBase",
    "A31A31DYNRightCh1Overall": "NonMotorBearing",
    "A31A31DYNRightCh2Overall": "NonMotorBase",
    "A31A31DYNRightCh3Overall": "UpperMotor",
    "A31Cps_ActSpd": "ActualSpeed",
}

VIBRATION_COLUMNS = [
    "LowerMotor",
    "UpperGearbox",
    "LowerGearbox",
    "MotorSideBearing",
    "MotorSideBase",
    "NonMotorBearing",
    "NonMotorBase",
    "UpperMotor",
]

SPEED_COLUMN = "ActualSpeed"

RUNNING_SPEED_THRESHOLD = 5.0
STARTUP_EXCLUSION_SECONDS = 120
SHUTDOWN_EXCLUSION_SECONDS = 120

WINDOW_SECONDS = 300
WINDOW_STEP_SECONDS = 60
MIN_WINDOW_COMPLETENESS = 0.90
MAX_INTERNAL_GAP_SECONDS = 5

N_OPERATING_REGIMES = 2
ISOLATION_CONTAMINATION = 0.005
MIN_CONSECUTIVE_ANOMALOUS_WINDOWS = 3



## 4. Flexible AVEVA import

This importer accepts `.xlsx`, `.xls`, and `.csv` files.

It attempts to identify:

- Timestamp column
- Tag-name column
- Numeric value column

Extra AVEVA metadata columns are ignored.


In [ ]:

TIMESTAMP_CANDIDATES = [
    "DateTime", "Datetime", "Timestamp", "TimeStamp", "Time", "Date"
]

TAG_CANDIDATES = [
    "TagName", "Tag Name", "Tag", "Name"
]

VALUE_CANDIDATES = [
    "Value", "AnalogValue", "ProcessValue", "Val"
]


def _find_column(columns, candidates):
    normalized = {str(c).strip().lower(): c for c in columns}

    for candidate in candidates:
        match = normalized.get(candidate.lower())
        if match is not None:
            return match

    return None


def read_export_file(file_path):
    file_path = Path(file_path)

    if file_path.suffix.lower() in [".xlsx", ".xls"]:
        raw = pd.read_excel(file_path)
    elif file_path.suffix.lower() == ".csv":
        raw = pd.read_csv(file_path)
    else:
        raise ValueError(f"Unsupported file type: {file_path.suffix}")

    timestamp_col = _find_column(raw.columns, TIMESTAMP_CANDIDATES)
    tag_col = _find_column(raw.columns, TAG_CANDIDATES)
    value_col = _find_column(raw.columns, VALUE_CANDIDATES)

    # Fallback for exports whose headers are missing or generic.
    if timestamp_col is None and len(raw.columns) >= 1:
        timestamp_col = raw.columns[0]
    if tag_col is None and len(raw.columns) >= 2:
        tag_col = raw.columns[1]
    if value_col is None and len(raw.columns) >= 3:
        value_col = raw.columns[2]

    subset = raw[[timestamp_col, tag_col, value_col]].copy()
    subset.columns = ["Timestamp", "TagName", "Value"]

    subset["Timestamp"] = pd.to_datetime(subset["Timestamp"], errors="coerce")
    subset["TagName"] = subset["TagName"].astype(str).str.strip()
    subset["Value"] = pd.to_numeric(subset["Value"], errors="coerce")

    subset = subset.dropna(subset=["Timestamp", "TagName", "Value"])
    subset = subset[subset["TagName"].isin(TAG_MAP.keys())].copy()

    subset["ModelColumn"] = subset["TagName"].map(TAG_MAP)
    subset["SourceFile"] = file_path.name

    return subset


def list_data_files(folder):
    folder = Path(folder)
    files = []
    for pattern in ["*.xlsx", "*.xls", "*.csv"]:
        files.extend(folder.glob(pattern))
    return sorted(set(files))


def import_folder(folder):
    files = list_data_files(folder)

    if not files:
        print(f"No data files found in {Path(folder).resolve()}")
        return pd.DataFrame()

    pieces = []

    for file_path in files:
        try:
            part = read_export_file(file_path)
            pieces.append(part)
            print(f"Imported {file_path.name}: {len(part):,} usable rows")
        except Exception as exc:
            print(f"FAILED {file_path.name}: {exc}")

    if not pieces:
        return pd.DataFrame()

    combined = pd.concat(pieces, ignore_index=True)
    combined = combined.sort_values(["Timestamp", "ModelColumn"])

    return combined


## 5. Import historical files incrementally

In [ ]:

historical_long = import_folder(HISTORICAL_DATA_DIR)

if not historical_long.empty:
    print("Historical date range:")
    print(historical_long["Timestamp"].min(), "to", historical_long["Timestamp"].max())
    display(historical_long.head(20))



## 6. Convert long AVEVA data to wide format

Multiple files can overlap. For duplicate timestamp/tag combinations, the last available value is retained.


In [ ]:

def long_to_wide(long_df):
    if long_df.empty:
        return pd.DataFrame()

    clean = (
        long_df.sort_values(["Timestamp", "ModelColumn", "SourceFile"])
               .drop_duplicates(
                   subset=["Timestamp", "ModelColumn"],
                   keep="last"
               )
    )

    wide = clean.pivot(
        index="Timestamp",
        columns="ModelColumn",
        values="Value"
    ).sort_index()

    wide.columns.name = None
    wide = wide.reset_index()

    required = [SPEED_COLUMN, *VIBRATION_COLUMNS]

    for column in required:
        if column not in wide.columns:
            wide[column] = np.nan

    wide = wide[["Timestamp", SPEED_COLUMN, *VIBRATION_COLUMNS]]
    return wide


historical_wide = long_to_wide(historical_long)

if not historical_wide.empty:
    print("Wide-format rows:", f"{len(historical_wide):,}")
    display(historical_wide.head())


## 7. Data quality and one-second resampling

In [ ]:

def clean_and_resample(wide_df):
    if wide_df.empty:
        return pd.DataFrame()

    data = (
        wide_df.drop_duplicates("Timestamp", keep="last")
               .sort_values("Timestamp")
               .set_index("Timestamp")
    )

    full_index = pd.date_range(
        data.index.min().floor("s"),
        data.index.max().ceil("s"),
        freq="1s",
    )

    data = data.reindex(full_index)
    data.index.name = "Timestamp"

    numeric_columns = [SPEED_COLUMN, *VIBRATION_COLUMNS]

    # Only interpolate short gaps.
    data[numeric_columns] = data[numeric_columns].interpolate(
        method="time",
        limit=5,
        limit_direction="both",
    )

    return data.reset_index()


historical_clean = clean_and_resample(historical_wide)

if not historical_clean.empty:
    print("Resampled rows:", f"{len(historical_clean):,}")
    print("Missing values by column:")
    display(historical_clean.isna().sum().to_frame("Missing"))


## 8. Identify continuous machine runs

In [ ]:

def identify_runs(data):
    data = data.copy()

    data["is_running_raw"] = data[SPEED_COLUMN] > RUNNING_SPEED_THRESHOLD

    state_change = data["is_running_raw"].ne(
        data["is_running_raw"].shift(fill_value=False)
    )
    data["state_group"] = state_change.cumsum()
    data["run_id"] = np.nan

    run_counter = 0

    for _, group in data.groupby("state_group"):
        if bool(group["is_running_raw"].iloc[0]):
            run_counter += 1
            data.loc[group.index, "run_id"] = run_counter

    data["seconds_from_run_start"] = np.nan
    data["seconds_to_run_end"] = np.nan

    for run_id, group in data.dropna(subset=["run_id"]).groupby("run_id"):
        idx = group.index
        start_time = group["Timestamp"].iloc[0]
        end_time = group["Timestamp"].iloc[-1]

        data.loc[idx, "seconds_from_run_start"] = (
            group["Timestamp"] - start_time
        ).dt.total_seconds().values

        data.loc[idx, "seconds_to_run_end"] = (
            end_time - group["Timestamp"]
        ).dt.total_seconds().values

    data["steady_running"] = (
        data["is_running_raw"]
        & (data["seconds_from_run_start"] >= STARTUP_EXCLUSION_SECONDS)
        & (data["seconds_to_run_end"] >= SHUTDOWN_EXCLUSION_SECONDS)
    )

    return data


historical_runs = identify_runs(historical_clean)

if not historical_runs.empty:
    print("Detected runs:", int(historical_runs["run_id"].nunique()))
    print("Steady-running samples:", f"{int(historical_runs['steady_running'].sum()):,}")


## 9. Feature engineering for five-minute windows

In [ ]:

def linear_slope(values):
    values = np.asarray(values, dtype=float)
    valid = np.isfinite(values)

    if valid.sum() < 3:
        return np.nan

    x = np.arange(len(values), dtype=float)
    return np.polyfit(x[valid], values[valid], 1)[0]


def build_windows(data):
    rows = []
    steady = data[data["steady_running"]].copy()

    for run_id, run in steady.groupby("run_id"):
        run = run.sort_values("Timestamp").reset_index(drop=True)

        if len(run) < WINDOW_SECONDS:
            continue

        current = run["Timestamp"].iloc[0]
        last_time = run["Timestamp"].iloc[-1]

        while current + pd.Timedelta(seconds=WINDOW_SECONDS - 1) <= last_time:
            end = current + pd.Timedelta(seconds=WINDOW_SECONDS - 1)

            block = run[
                (run["Timestamp"] >= current)
                & (run["Timestamp"] <= end)
            ].copy()

            completeness = len(block) / WINDOW_SECONDS

            if completeness < MIN_WINDOW_COMPLETENESS:
                current += pd.Timedelta(seconds=WINDOW_STEP_SECONDS)
                continue

            gaps = block["Timestamp"].diff().dt.total_seconds()

            if gaps.max() > MAX_INTERNAL_GAP_SECONDS:
                current += pd.Timedelta(seconds=WINDOW_STEP_SECONDS)
                continue

            if block[[SPEED_COLUMN, *VIBRATION_COLUMNS]].isna().any().any():
                current += pd.Timedelta(seconds=WINDOW_STEP_SECONDS)
                continue

            row = {
                "run_id": int(run_id),
                "window_start": current,
                "window_end": end,
                "window_center": current + pd.Timedelta(seconds=WINDOW_SECONDS / 2),
                "speed_mean": block[SPEED_COLUMN].mean(),
                "speed_std": block[SPEED_COLUMN].std(),
                "speed_min": block[SPEED_COLUMN].min(),
                "speed_max": block[SPEED_COLUMN].max(),
                "speed_slope": linear_slope(block[SPEED_COLUMN]),
                "seconds_from_run_start": block["seconds_from_run_start"].median(),
            }

            means = {}

            for sensor in VIBRATION_COLUMNS:
                values = block[sensor].to_numpy(dtype=float)
                means[sensor] = values.mean()

                row[f"{sensor}_mean"] = values.mean()
                row[f"{sensor}_median"] = np.median(values)
                row[f"{sensor}_std"] = values.std()
                row[f"{sensor}_min"] = values.min()
                row[f"{sensor}_max"] = values.max()
                row[f"{sensor}_range"] = np.ptp(values)
                row[f"{sensor}_slope"] = linear_slope(values)
                row[f"{sensor}_delta"] = values[-30:].mean() - values[:30].mean()
                row[f"{sensor}_p90"] = np.percentile(values, 90)
                row[f"{sensor}_diff_std"] = np.diff(values).std()

            eps = 1e-9
            total = sum(means.values()) + eps

            for sensor, value in means.items():
                row[f"{sensor}_share"] = value / total

            row["ratio_motor_top_bottom"] = (
                means["UpperMotor"] / (means["LowerMotor"] + eps)
            )
            row["ratio_gearbox_top_bottom"] = (
                means["UpperGearbox"] / (means["LowerGearbox"] + eps)
            )
            row["ratio_motor_bearing_to_base"] = (
                means["MotorSideBearing"] / (means["MotorSideBase"] + eps)
            )
            row["ratio_nonmotor_bearing_to_base"] = (
                means["NonMotorBearing"] / (means["NonMotorBase"] + eps)
            )
            row["ratio_motor_to_nonmotor_bearing"] = (
                means["MotorSideBearing"] / (means["NonMotorBearing"] + eps)
            )

            row["motor_average"] = np.mean([
                means["UpperMotor"],
                means["LowerMotor"],
            ])
            row["gearbox_average"] = np.mean([
                means["UpperGearbox"],
                means["LowerGearbox"],
            ])
            row["bearing_average"] = np.mean([
                means["MotorSideBearing"],
                means["NonMotorBearing"],
            ])
            row["base_average"] = np.mean([
                means["MotorSideBase"],
                means["NonMotorBase"],
            ])
            row["bearing_minus_base"] = row["bearing_average"] - row["base_average"]
            row["gearbox_minus_motor"] = row["gearbox_average"] - row["motor_average"]

            rows.append(row)
            current += pd.Timedelta(seconds=WINDOW_STEP_SECONDS)

    return pd.DataFrame(rows)


historical_windows = build_windows(historical_runs)

print("Feature windows:", f"{len(historical_windows):,}")
display(historical_windows.head())



# Phase 1 — Learn Normal and Detect Anomalies

Use this phase as soon as you have one mostly healthy month.

It learns:

- Expected vibration versus line speed
- Two recurring operating regimes
- A separate normal baseline for each regime
- Persistent anomaly events


## 10. Train or load Phase 1 models

In [ ]:

TRAIN_PHASE_1 = True

SPEED_CONTEXT_COLUMNS = [
    "speed_mean",
    "speed_std",
    "speed_min",
    "speed_max",
    "speed_slope",
    "seconds_from_run_start",
]

if historical_windows.empty:
    raise ValueError("No windows are available. Add data files first.")

speed_models = {}
speed_residual_columns = []

if TRAIN_PHASE_1:
    for sensor in VIBRATION_COLUMNS:
        model = HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.05,
            max_leaf_nodes=20,
            l2_regularization=1.0,
            random_state=RANDOM_SEED,
        )

        target = f"{sensor}_mean"
        model.fit(historical_windows[SPEED_CONTEXT_COLUMNS], historical_windows[target])

        historical_windows[f"{sensor}_expected"] = model.predict(
            historical_windows[SPEED_CONTEXT_COLUMNS]
        )
        historical_windows[f"{sensor}_speed_residual"] = (
            historical_windows[target]
            - historical_windows[f"{sensor}_expected"]
        )

        speed_models[sensor] = model
        speed_residual_columns.append(f"{sensor}_speed_residual")

    joblib.dump(speed_models, MODELS_DIR / "speed_models.joblib")
else:
    speed_models = joblib.load(MODELS_DIR / "speed_models.joblib")

    for sensor in VIBRATION_COLUMNS:
        target = f"{sensor}_mean"
        historical_windows[f"{sensor}_expected"] = speed_models[sensor].predict(
            historical_windows[SPEED_CONTEXT_COLUMNS]
        )
        historical_windows[f"{sensor}_speed_residual"] = (
            historical_windows[target]
            - historical_windows[f"{sensor}_expected"]
        )
        speed_residual_columns.append(f"{sensor}_speed_residual")


## 11. Detect operating regimes automatically

In [ ]:

REGIME_FEATURE_COLUMNS = (
    speed_residual_columns
    + [f"{sensor}_share" for sensor in VIBRATION_COLUMNS]
    + [
        "ratio_motor_top_bottom",
        "ratio_gearbox_top_bottom",
        "ratio_motor_bearing_to_base",
        "ratio_nonmotor_bearing_to_base",
        "ratio_motor_to_nonmotor_bearing",
        "bearing_minus_base",
        "gearbox_minus_motor",
    ]
)

if TRAIN_PHASE_1:
    regime_scaler = RobustScaler()
    X_regime = regime_scaler.fit_transform(
        historical_windows[REGIME_FEATURE_COLUMNS]
    )

    regime_model = GaussianMixture(
        n_components=2,
        covariance_type="full",
        n_init=20,
        random_state=RANDOM_SEED,
    )

    labels = regime_model.fit_predict(X_regime)

    joblib.dump(regime_scaler, MODELS_DIR / "regime_scaler.joblib")
    joblib.dump(regime_model, MODELS_DIR / "regime_model.joblib")
else:
    regime_scaler = joblib.load(MODELS_DIR / "regime_scaler.joblib")
    regime_model = joblib.load(MODELS_DIR / "regime_model.joblib")

    X_regime = regime_scaler.transform(
        historical_windows[REGIME_FEATURE_COLUMNS]
    )
    labels = regime_model.predict(X_regime)

historical_windows["detected_regime"] = [
    f"regime_{label}" for label in labels
]
historical_windows["regime_confidence"] = (
    regime_model.predict_proba(X_regime).max(axis=1)
)

display(historical_windows["detected_regime"].value_counts())


## 12. Train anomaly detectors

In [ ]:

ANOMALY_FEATURE_COLUMNS = (
    SPEED_CONTEXT_COLUMNS
    + [
        f"{sensor}_{stat}"
        for sensor in VIBRATION_COLUMNS
        for stat in [
            "mean", "median", "std", "max",
            "range", "slope", "delta", "p90", "diff_std"
        ]
    ]
    + speed_residual_columns
    + [f"{sensor}_share" for sensor in VIBRATION_COLUMNS]
    + [
        "ratio_motor_top_bottom",
        "ratio_gearbox_top_bottom",
        "ratio_motor_bearing_to_base",
        "ratio_nonmotor_bearing_to_base",
        "ratio_motor_to_nonmotor_bearing",
        "motor_average",
        "gearbox_average",
        "bearing_average",
        "base_average",
        "bearing_minus_base",
        "gearbox_minus_motor",
    ]
)

phase1_assets = {}

if TRAIN_PHASE_1:
    for regime, group in historical_windows.groupby("detected_regime"):
        scaler = RobustScaler()
        X = scaler.fit_transform(group[ANOMALY_FEATURE_COLUMNS])

        model = IsolationForest(
            n_estimators=500,
            contamination=ISOLATION_CONTAMINATION,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )
        model.fit(X)

        phase1_assets[regime] = {
            "scaler": scaler,
            "model": model,
        }

    joblib.dump(phase1_assets, MODELS_DIR / "phase1_anomaly_assets.joblib")
else:
    phase1_assets = joblib.load(MODELS_DIR / "phase1_anomaly_assets.joblib")

historical_windows["anomaly_score"] = np.nan
historical_windows["is_anomaly_raw"] = False

for regime, group in historical_windows.groupby("detected_regime"):
    asset = phase1_assets[regime]

    X = asset["scaler"].transform(
        group[ANOMALY_FEATURE_COLUMNS]
    )

    historical_windows.loc[group.index, "anomaly_score"] = (
        -asset["model"].score_samples(X)
    )

    historical_windows.loc[group.index, "is_anomaly_raw"] = (
        asset["model"].predict(X) == -1
    )


## 13. Persistent anomaly events

In [ ]:

historical_windows = historical_windows.sort_values(
    "window_center"
).reset_index(drop=True)

historical_windows["persistent_anomaly"] = False
historical_windows["anomaly_event_id"] = np.nan

event_counter = 0

for run_id, group in historical_windows.groupby("run_id"):
    idx = list(group.index)
    flags = historical_windows.loc[idx, "is_anomaly_raw"].to_numpy(dtype=bool)

    start = 0

    while start < len(flags):
        if not flags[start]:
            start += 1
            continue

        end = start + 1

        while end < len(flags) and flags[end]:
            end += 1

        if end - start >= MIN_CONSECUTIVE_ANOMALOUS_WINDOWS:
            event_counter += 1
            event_idx = idx[start:end]

            historical_windows.loc[event_idx, "persistent_anomaly"] = True
            historical_windows.loc[event_idx, "anomaly_event_id"] = event_counter

        start = end

print("Persistent anomaly events:", event_counter)


## 14. Phase 1 plots and exports

In [ ]:

plt.figure(figsize=(15, 5))
plt.plot(
    historical_windows["window_center"],
    historical_windows["anomaly_score"],
    linewidth=0.7,
)

flagged = historical_windows[historical_windows["persistent_anomaly"]]

plt.scatter(
    flagged["window_center"],
    flagged["anomaly_score"],
    s=25,
    label="Persistent anomaly",
)

plt.title("Historical Anomaly Score")
plt.xlabel("Time")
plt.ylabel("Higher = more unusual")
plt.legend()
plt.tight_layout()
plt.show()

historical_windows.to_csv(
    PROCESSED_DIR / "historical_window_scores.csv",
    index=False,
)

historical_windows.sort_values(
    "anomaly_score",
    ascending=False
).head(500).to_csv(
    OUTPUTS_DIR / "phase1_top_500_anomaly_windows.csv",
    index=False,
)

print("Phase 1 outputs saved.")



# Phase 2 — Add Known Failures

You can begin this phase later.

Create:

```text
strander_project/data/failures.csv
```

with columns:

```text
failure_id,failure_time,failure_type,affected_area,description,time_accuracy
F001,2026-07-17 14:32:00,Bearing Failure,Motor-side bearing,Bearing replaced after drive fault,Exact
```

Approximate event times are acceptable. Enter the best information available.


In [ ]:

def create_failure_template():
    template = pd.DataFrame(columns=[
        "failure_id",
        "failure_time",
        "failure_type",
        "affected_area",
        "description",
        "time_accuracy",
    ])

    template.to_csv(FAILURE_FILE, index=False)
    print("Created:", FAILURE_FILE.resolve())


if not FAILURE_FILE.exists():
    create_failure_template()


## 15. Load failure events and label pre-failure windows

In [ ]:

LOOKBACK_HOURS = 72
EXCLUDE_AFTER_FAILURE_HOURS = 4

if FAILURE_FILE.exists():
    failures = pd.read_csv(FAILURE_FILE)
    failures["failure_time"] = pd.to_datetime(
        failures["failure_time"],
        errors="coerce"
    )
    failures = failures.dropna(subset=["failure_time"])
else:
    failures = pd.DataFrame()

labeled_windows = historical_windows.copy()
labeled_windows["failure_label"] = "normal"
labeled_windows["failure_id"] = pd.NA
labeled_windows["hours_to_failure"] = np.nan

for _, failure in failures.iterrows():
    failure_time = failure["failure_time"]

    mask = (
        (labeled_windows["window_center"] >= failure_time - pd.Timedelta(hours=LOOKBACK_HOURS))
        & (labeled_windows["window_center"] < failure_time)
    )

    labeled_windows.loc[mask, "failure_label"] = failure["failure_type"]
    labeled_windows.loc[mask, "failure_id"] = failure["failure_id"]

    labeled_windows.loc[mask, "hours_to_failure"] = (
        failure_time - labeled_windows.loc[mask, "window_center"]
    ).dt.total_seconds() / 3600

print("Failure events loaded:", len(failures))
display(
    labeled_windows["failure_label"].value_counts()
)



## 16. Build failure-pattern signatures

Each confirmed failure creates a sequence-level signature from the preceding hours.

This first version summarizes:

- Residual vibration by sensor
- Anomaly score
- Sensor relationships
- Trend and variability
- Time-to-failure progression

Later, this can be upgraded to a neural sequence model after enough failures are available.


In [ ]:

PATTERN_FEATURE_COLUMNS = (
    speed_residual_columns
    + [
        "anomaly_score",
        "ratio_motor_top_bottom",
        "ratio_gearbox_top_bottom",
        "ratio_motor_bearing_to_base",
        "ratio_nonmotor_bearing_to_base",
        "ratio_motor_to_nonmotor_bearing",
        "bearing_minus_base",
        "gearbox_minus_motor",
    ]
    + [f"{sensor}_slope" for sensor in VIBRATION_COLUMNS]
    + [f"{sensor}_std" for sensor in VIBRATION_COLUMNS]
)

failure_signatures = []

for failure_id, group in labeled_windows.dropna(
    subset=["failure_id"]
).groupby("failure_id"):

    failure_row = failures[
        failures["failure_id"] == failure_id
    ].iloc[0]

    signature = {
        "failure_id": failure_id,
        "failure_type": failure_row["failure_type"],
        "affected_area": failure_row.get("affected_area", ""),
        "failure_time": failure_row["failure_time"],
        "window_count": len(group),
    }

    for column in PATTERN_FEATURE_COLUMNS:
        signature[f"{column}__mean"] = group[column].mean()
        signature[f"{column}__max"] = group[column].max()
        signature[f"{column}__last6h"] = group.loc[
            group["hours_to_failure"] <= 6, column
        ].mean()

    failure_signatures.append(signature)

failure_signatures = pd.DataFrame(failure_signatures)

if not failure_signatures.empty:
    failure_signatures.to_csv(
        PROCESSED_DIR / "failure_signatures.csv",
        index=False,
    )
    display(failure_signatures.head())
else:
    print("No failure signatures yet. Add failures.csv entries when available.")



# Phase 3 — Score Current Data and Compare to Previous Failures

Place one or more new Excel/CSV exports in:

```text
strander_project/data/current/
```

This phase:

1. Runs the current data through the saved Phase 1 baseline
2. Detects current anomalies
3. Builds a current pattern signature
4. Compares it to historical failure signatures
5. Reports the closest prior failure


## 17. Import and process current data

In [ ]:

current_long = import_folder(CURRENT_DATA_DIR)
current_wide = long_to_wide(current_long)
current_clean = clean_and_resample(current_wide)
current_runs = identify_runs(current_clean)
current_windows = build_windows(current_runs)

print("Current windows:", len(current_windows))


## 18. Apply saved Phase 1 models to current data

In [ ]:

def apply_phase1_models(windows):
    windows = windows.copy()

    if windows.empty:
        return windows

    speed_models = joblib.load(MODELS_DIR / "speed_models.joblib")
    regime_scaler = joblib.load(MODELS_DIR / "regime_scaler.joblib")
    regime_model = joblib.load(MODELS_DIR / "regime_model.joblib")
    anomaly_assets = joblib.load(MODELS_DIR / "phase1_anomaly_assets.joblib")

    residual_columns = []

    for sensor in VIBRATION_COLUMNS:
        target = f"{sensor}_mean"
        expected = speed_models[sensor].predict(
            windows[SPEED_CONTEXT_COLUMNS]
        )

        windows[f"{sensor}_expected"] = expected
        windows[f"{sensor}_speed_residual"] = (
            windows[target] - expected
        )

        residual_columns.append(f"{sensor}_speed_residual")

    X_regime = regime_scaler.transform(
        windows[REGIME_FEATURE_COLUMNS]
    )

    labels = regime_model.predict(X_regime)

    windows["detected_regime"] = [
        f"regime_{label}" for label in labels
    ]
    windows["regime_confidence"] = (
        regime_model.predict_proba(X_regime).max(axis=1)
    )

    windows["anomaly_score"] = np.nan
    windows["is_anomaly_raw"] = False

    for regime, group in windows.groupby("detected_regime"):
        if regime not in anomaly_assets:
            continue

        asset = anomaly_assets[regime]
        X = asset["scaler"].transform(
            group[ANOMALY_FEATURE_COLUMNS]
        )

        windows.loc[group.index, "anomaly_score"] = (
            -asset["model"].score_samples(X)
        )

        windows.loc[group.index, "is_anomaly_raw"] = (
            asset["model"].predict(X) == -1
        )

    return windows


current_scored = apply_phase1_models(current_windows)
display(current_scored.head())


## 19. Compare current pattern to prior failures

In [ ]:

def build_single_signature(group):
    signature = {}

    for column in PATTERN_FEATURE_COLUMNS:
        signature[f"{column}__mean"] = group[column].mean()
        signature[f"{column}__max"] = group[column].max()

        recent_cutoff = group["window_center"].max() - pd.Timedelta(hours=6)
        signature[f"{column}__last6h"] = group.loc[
            group["window_center"] >= recent_cutoff,
            column
        ].mean()

    return pd.Series(signature)


if current_scored.empty:
    print("No current data available.")
elif failure_signatures.empty:
    print("Current data was scored, but no historical failure signatures exist yet.")
else:
    current_signature = build_single_signature(current_scored)

    signature_columns = [
        column for column in failure_signatures.columns
        if column in current_signature.index
    ]

    reference = failure_signatures[signature_columns].copy()
    reference = reference.fillna(reference.median(numeric_only=True)).fillna(0)

    current_vector = current_signature[signature_columns].fillna(0).to_numpy().reshape(1, -1)

    similarity_scaler = StandardScaler()
    reference_scaled = similarity_scaler.fit_transform(reference)
    current_scaled = similarity_scaler.transform(current_vector)

    similarities = cosine_similarity(
        current_scaled,
        reference_scaled
    ).flatten()

    comparison = failure_signatures[
        [
            "failure_id",
            "failure_type",
            "affected_area",
            "failure_time",
            "window_count",
        ]
    ].copy()

    comparison["similarity"] = similarities
    comparison = comparison.sort_values(
        "similarity",
        ascending=False
    )

    display(comparison.head(10))

    comparison.to_csv(
        OUTPUTS_DIR / "phase3_current_failure_similarity.csv",
        index=False,
    )


## 20. Current condition summary

In [ ]:

if not current_scored.empty:
    summary = {
        "analysis_start": current_scored["window_start"].min(),
        "analysis_end": current_scored["window_end"].max(),
        "window_count": len(current_scored),
        "maximum_anomaly_score": current_scored["anomaly_score"].max(),
        "mean_anomaly_score": current_scored["anomaly_score"].mean(),
        "raw_anomaly_windows": int(current_scored["is_anomaly_raw"].sum()),
        "dominant_regime": current_scored["detected_regime"].mode().iloc[0],
        "mean_regime_confidence": current_scored["regime_confidence"].mean(),
    }

    display(pd.DataFrame([summary]))

    plt.figure(figsize=(15, 5))
    plt.plot(
        current_scored["window_center"],
        current_scored["anomaly_score"],
        linewidth=0.9,
    )

    abnormal = current_scored[current_scored["is_anomaly_raw"]]

    plt.scatter(
        abnormal["window_center"],
        abnormal["anomaly_score"],
        s=30,
        label="Abnormal window",
    )

    plt.title("Current Data Anomaly Score")
    plt.xlabel("Time")
    plt.ylabel("Higher = more unusual")
    plt.legend()
    plt.tight_layout()
    plt.show()

    current_scored.to_csv(
        OUTPUTS_DIR / "current_scored_windows.csv",
        index=False,
    )



# Recommended Incremental Workflow

## Right now

1. Put the July 1 Excel export into:
   `strander_project/data/historical/`
2. Run Sections 1–9.
3. Confirm the import and tag mapping.
4. Add several mostly healthy days or one healthy month.
5. Set `TRAIN_PHASE_1 = True`.
6. Run Phase 1.

## As you collect more months

- Add each Excel/CSV export to `data/historical/`
- Re-run the import and preprocessing sections
- Do not retrain automatically every time
- Retrain only after reviewing whether the new period is healthy

## As you find crash dates

- Enter them into `strander_project/data/failures.csv`
- Run Phase 2
- Check whether the preceding 6–72 hours contain repeatable patterns

## For today's data

- Remove old files from `data/current/`
- Add today's export
- Run Phase 3
- Review:
  - anomaly score
  - operating regime
  - closest historical failure
  - similarity
  - affected sensor area

---

# Important caution

Similarity to a prior failure is evidence for inspection, not proof that the same failure will occur.

Reliable predictive performance will require:

- several confirmed failures
- accurate event times
- healthy comparison periods
- review of false alarms
- avoiding model retraining on damaged operation
